In [1]:
import glob
rirfiles = glob.glob("/data4/Henri/j3/framewiseSpeakerCounting/databases/brudex/rir/rev_low/*BTE_IE.pt")

In [2]:
import torch

In [3]:
rir8000 = []
rir16000 = []
for rirfile in rirfiles:
    if "8000" in rirfile:
        rir8000.append(torch.load(rirfile, weights_only=False))
    elif "16000" in rirfile:
        rir16000.append(torch.load(rirfile, weights_only=False))
        

In [8]:
from utilities import rir2rtf, STFTtransform, print_structure

transform8000 = STFTtransform(
    frame_length=64e-3,
    frame_shift=16e-3,
    sampling_frequency=8000,
    window_type="sqrt-hann"
    )
transform16000 = STFTtransform(
    frame_length=64e-3,
    frame_shift=16e-3,
    sampling_frequency=16000,
    window_type="sqrt-hann"
    )

rtf8000 = torch.cat([rir2rtf(rir, transform8000) for rir in rir8000], dim=-1)
rtf16000 = torch.cat([rir2rtf(rir, transform16000) for rir in rir16000], dim=-1)

print_structure(rtf8000)
print_structure(rtf16000)

Tensor (257, 1, 6, 12)
Tensor (513, 1, 6, 12)


In [5]:
from utilities import hermitian_angle
a8000 = rtf8000.transpose(-1,-3).unsqueeze(-3)
b8000 = rtf8000.transpose(-1,-3).unsqueeze(-4)
HA8000 = hermitian_angle(a8000, b8000, dim=-2)
print(f"a-shape: {a8000.shape},\nb-shape: {b8000.shape},\nHA-shape: {HA8000.shape}")
a16000 = rtf16000.transpose(-1,-3).unsqueeze(-3)
b16000 = rtf16000.transpose(-1,-3).unsqueeze(-4)
HA16000 = hermitian_angle(a16000, b16000, dim=-2)
print(f"a-shape: {a16000.shape},\nb-shape: {b16000.shape},\nHA-shape: {HA16000.shape}")

a-shape: torch.Size([257, 12, 1, 6, 1]),
b-shape: torch.Size([257, 1, 12, 6, 1]),
HA-shape: torch.Size([257, 12, 12, 1, 1])
a-shape: torch.Size([513, 12, 1, 6, 1]),
b-shape: torch.Size([513, 1, 12, 6, 1]),
HA-shape: torch.Size([513, 12, 12, 1, 1])


In [6]:
MHA8000 = HA8000.mean(dim=0).squeeze()
MHA16000 = HA16000.mean(dim=0).squeeze()

In [7]:
rows, cols = torch.tril_indices(12, 12, offset=-1)

# Extract values
MHA8000_vals = MHA8000[rows, cols]
MHA16000_vals = MHA16000[rows, cols]
print(f"MHA8000 values: {MHA8000_vals}, max: {MHA8000_vals.max()}, min: {MHA8000_vals.min()}")
print(f"MHA16000 values: {MHA16000_vals}, max: {MHA16000_vals.max()}, min: {MHA16000_vals.min()}")

MHA8000 values: tensor([0.7768, 1.0002, 1.0376, 0.6099, 0.6281, 0.9821, 1.0133, 1.0952, 0.6748,
        1.1167, 0.8132, 0.9218, 0.8419, 0.8406, 0.8592, 1.0894, 0.9822, 0.8173,
        1.0361, 0.9706, 1.0287, 1.0794, 1.0448, 0.8540, 1.0781, 0.6487, 1.0127,
        0.8344, 0.9828, 0.8271, 0.9713, 0.9290, 0.9912, 0.7240, 0.8945, 0.9471,
        0.7298, 0.6710, 0.9924, 0.5723, 1.0746, 0.8877, 0.9901, 1.0404, 0.8896,
        0.8132, 0.7855, 1.0873, 0.9150, 1.0469, 0.9899, 0.9672, 0.9898, 0.8711,
        0.8233, 1.0307, 1.0589, 0.7643, 1.0756, 0.5400, 0.9000, 0.8841, 0.6880,
        0.9490, 1.0396, 1.0098]), max: 1.1167024374008179, min: 0.5399850606918335
MHA16000 values: tensor([1.1018, 0.7544, 1.1458, 1.1536, 0.6692, 1.1772, 1.2108, 0.9536, 1.1857,
        0.8677, 0.9012, 1.0305, 0.9557, 1.1289, 1.1465, 0.8851, 1.1661, 0.6326,
        1.1856, 1.1439, 1.0520, 1.1419, 1.1377, 1.0850, 1.0564, 0.8921, 1.0038,
        1.0120, 1.0340, 1.2127, 0.8958, 1.1907, 1.0993, 1.1759, 0.7726, 0.8780,
    

In [9]:
PI

NameError: name 'PI' is not defined